# M1 求值器

用**树**这一数据结构管理中间结果，并管理各算子路径。

目前是用模拟数据，看逻辑是否能跑通。


In [ ]:
import numpy as np
import pandas as pd

from src.gp.engine import random_tree
from src.gp.evaluator import evaluate, to_wide
from src.gp.tree import Node

In [4]:
# 造一个迷你长表：3 只股票 x 10天
dates = pd.date_range("2024-01-01", periods=10)
codes = ["000001", "000002", "6000000"]
idx = pd.MultiIndex.from_product([dates, codes], names=["date", "code"])
rng = np.random.default_rng(0)
prices = pd.DataFrame(
    {c: rng.random(len(idx)) * 100 for c in ["open", "high", "low", "close", "volume", "amount"]},
    index=idx,
)

wide = to_wide(prices=prices)
assert wide["close"].shape == (10, 3)
wide

{'open': code           000001     000002    6000000
 date                                       
 2024-01-01  63.696169  26.978671   4.097352
 2024-01-02   1.652764  81.327024  91.275558
 2024-01-03  60.663578  72.949656  54.362499
 2024-01-04  93.507242  81.585355   0.273850
 2024-01-05  85.740428   3.358558  72.965545
 2024-01-06  17.565562  86.317892  54.146122
 2024-01-07  29.971189  42.268722   2.831967
 2024-01-08  12.428328  67.062441  64.718951
 2024-01-09  61.538511  38.367755  99.720994
 2024-01-10  98.083534  68.554198  65.045928,
 'high': code           000001     000002    6000000
 date                                       
 2024-01-01  68.844673  38.892142  13.509651
 2024-01-02  72.148834  52.535432  31.024188
 2024-01-03  48.583536  88.948783  93.404352
 2024-01-04  35.779520  57.152983  32.186939
 2024-01-05  59.430003  33.791123  39.161900
 2024-01-06  89.027435  22.715759  62.318714
 2024-01-07   8.401534  83.264415  78.709831
 2024-01-08  23.936944  87.648423   5.

In [5]:
# 手搓一棵树
tree = Node(
    "rank",
    1,
    [
        Node(
            "div",
            2,
            [
                Node("close", 0, value="close"),
                Node("ts_mean", 1, [Node("volume", 0, value="volume")], value=5),
            ],
        )
    ],
)

factor = evaluate(tree, wide=wide)
print(factor.shape)
print(factor.tail())

(10, 3)
code          000001    000002   6000000
date                                    
2024-01-06  0.666667  0.333333  1.000000
2024-01-07  0.333333  0.666667  1.000000
2024-01-08  0.666667  1.000000  0.333333
2024-01-09  0.666667  0.333333  1.000000
2024-01-10  0.333333  1.000000  0.666667


In [6]:
# 随机树压力测试：连跑 200 棵树都不出错，保证鲁棒性
for _ in range(200):
    evaluate(random_tree(max_depth=5, min_depth=2), wide)
print("OK: 200 棵随机数全部求值成功")

OK: 200 棵随机数全部求值成功
